# Provenance Agent - Overall Workflow (PaleoPCAlite)

This notebook shows the provenance agent the way an end user meets it, on the canonical example `testing/paleoPCAlite.ipynb` (the LiPDGraph pathway). It covers **both** software and dataset citations, and shows all three ways to invoke the agent, most-recommended first:

1. **`%provenance` magic** - the intended notebook-native experience.
2. **`agent.run`** - the same requests, programmatically.
3. **The direct tools** - `cite_software` / `cite_data`, exposing the arguments the higher layers hide.

For the software building blocks see `workflow.ipynb`; for the data building blocks see `testing/data_workflow.ipynb`.

## Setup

Add `src/` to the path so the agent modules import.

In [ ]:
import sys
sys.path.insert(0, '../src')

The data workflow injects retrieval cells **into the target notebook**, and the magic / `agent.run` layers write in place. So we work on a throwaway copy of the example and leave the original untouched.

In [ ]:
import shutil

SOURCE = 'testing/paleoPCAlite.ipynb'
DEMO = 'testing/paleoPCAlite_demo.ipynb'
shutil.copy(SOURCE, DEMO)
print('working copy:', DEMO)

## 1. The headline: the `%provenance` magic

One natural-language line per request. Auto-detection of the current notebook is unreliable in VSCode, so we set the target once. Both requests now inject a cell into the target and report what they wrote (software: one metadata-DataFrame cell; datasets: one retrieval cell per dataset).

In [ ]:
%load_ext provenance

In [ ]:
%provenance_notebook testing/paleoPCAlite_demo.ipynb

In [ ]:
%provenance cite the software

In [ ]:
%provenance cite the datasets

To get the dataset citations themselves, open `testing/paleoPCAlite_demo.ipynb` and run the retrieval cells that were just appended - they reuse that notebook's already-loaded objects in its live kernel.

## 2. The natural-language router: `agent.run`

The same two requests, called programmatically. The model picks the tool and arguments; `run` returns one `{name, args, result}` dict per tool it called. Software `result` is the list of libraries the injected metadata cell covers; data `result` is the injected `[variable, tool]` pairs. Both inject **in place**, so both run against the copy.

In [ ]:
from agent import run

for call in run('cite the software', DEMO):
    print(call['result'])

In [ ]:
for call in run('cite the datasets', DEMO):
    print(call['name'], '->', call['result'])

## 3. The direct tools

The plain functions the two layers above call. Both inject a cell rather than returning text: `cite_software` a metadata-DataFrame cell, `cite_data` a retrieval cell per dataset. `output_path` lets either write to a copy instead of in place, and `fmt` chooses BibTeX vs APA for the data cell.

In [ ]:
from orchestrator import cite_software

libraries = cite_software(SOURCE, output_path='testing/paleoPCAlite_with_citations.ipynb')
print('injected a metadata cell for:', libraries)

In [ ]:
from orchestrator import cite_data

pairs = cite_data(SOURCE, output_path='testing/paleoPCAlite_with_citations.ipynb')
print('injected cells for:', pairs)

## What needs a live kernel and API keys

- **Routing** (`%provenance`, `agent.run`) calls Gemini to pick the tool. **Software citations no longer call Gemini**: `cite_software` injects a metadata-DataFrame cell built from the local `Citations/` files. APA rendering (Gemini) now applies only to the **data** cell, when `fmt='apa'`.
- **Dataset retrieval** runs in the target notebook's live kernel: the injected cells reuse its loaded objects, and LiPDGraph datasets are fetched from the LinkedEarth endpoint. So the agent *injects* the cells here, and you produce the citations by running them in the copy.